In [48]:
import warnings

import healpy as hp
import numpy as np
from scipy import interpolate
from scipy.stats import median_abs_deviation
from astropy import units as u
from astropy.coordinates import SkyCoord
from sklearn.cluster import KMeans
from copy import deepcopy
import rubin_sim.maf as maf
from rubin_sim.maf.metrics.uniformity_metrics import MultibandExgalM5

from rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric import StaticProbesFoMEmulatorMetric
from rubin_sim.maf.metrics.area_summary_metrics import AreaThresholdMetric
from rubin_sim.maf.metrics.base_metric import BaseMetric
from rubin_sim.maf.metrics.simple_metrics import RmsMetric
from rubin_scheduler.scheduler.utils import SkyAreaGenerator

In [38]:
# The next two lines are used later, when we want to impose a cut based on the level of dust to avoid the plane.
nside=64
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
year = 1
import pandas as pd
import sqlite3


In [44]:
# constraints
days = year*365.25
opsim_name='/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v3.6_10yrs.db'
run_name='baseline_v3.6_10yrs.db'

conn = sqlite3.connect(opsim_name)
d = pd.read_sql('select distinct(note) from observations', conn)
conn.close()
d


,note
0,


In [49]:
constraint_str = 'note not like "DD%" and night <= XX and note not like "twilight_near_sun" '
constraint_str = constraint_str.replace('XX','%d'%days)

count_metric = maf.CountMetric(col='observationStartMJD', metric_name='NVisits')
rms_metric = maf.RmsMetric(col='fiveSigmaDepth')
new_metric = MultibandExgalM5() 

slicer = maf.HealpixSubsetSlicer(nside=nside, hpid=np.where(map_labels == "lowdust")[0])

count_bundle = maf.MetricBundle(count_metric, slicer, constraint_str, run_name=run_name)
rms_bundle = maf.MetricBundle(new_metric, slicer, constraint_str + " and filter == 'r'", run_name=run_name)


Healpix slicer using NSIDE=64, approximate resolution 54.967783 arcminutes


In [50]:
g = maf.MetricBundleGroup({'count': count_bundle, 'rms': rms_bundle}, 
                          opsim_name, 
                          out_dir='tmp_out', 
                          verbose=True)


In [51]:
g.run_all()

Querying table None with constraint note not like "DD%" and night <= 365 and note not like "twilight_near_sun"  for columns ['fieldDec', 'rotSkyPos', 'observationStartMJD', 'fieldRA']
Found 184205 visits
Running:  ['count']
Processing slices:  42%|██████▎        | 20816/49152 [00:04<00:06, 4273.26it/s]
Completed metric generation.
Running reduce methods.
Running summary statistics.
Completed.
Querying table None with constraint note not like "DD%" and night <= 365 and note not like "twilight_near_sun"  and filter == 'r' for columns ['fieldDec', 'fieldRA', 'rotSkyPos', 'filter', 'fiveSigmaDepth']


/pscratch/sd/r/rhlozek/rubin_sim/rubin_sim/maf/slicers/base_spatial_slicer.py:118: UserWarning: Warning:  Loading maps but cache on.Should probably set use_cache=False in slicer.
  warnings.warn(
/pscratch/sd/r/rhlozek/rubin_sim/rubin_sim/maf/maps/dust_map.py:46: UserWarning: Slicer value of nside 64 different from map value 128, using slicer value
  warnings.warn(


Found 37184 visits
Running:  ['rms']
Processing slices:  42%|██████▎        | 20816/49152 [00:03<00:05, 5619.86it/s]
Completed metric generation.
Running reduce methods.
Running summary statistics.
Completed.


In [52]:
rms_bundle.plot()


/pscratch/sd/r/rhlozek/rubin_sim/rubin_sim/maf/plots/plot_handler.py:665: UserWarning: Cannot plot object metric values with this plotter.
  warnings.warn("Cannot plot object metric values with this plotter.")


{'SkyMap': None, 'Histogram': None}